<img src="https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/unifor-logo.png" width="400">
<br>
<b>
<font size="6" face="arial" color="blue">
    Graduação em Ciência da Computação
</font>
</b>
<br>
<b>
<font size="4" face="arial">
    Disciplina: Métodos Quantitativos em Computação
</font>
</b>

**Orientador: Prof. Me. Ricardo Carubbi** <br>
*Docente da Graduação e Pós-Graduação em Ciência de Dados e Inteligência Artificial*<br>
*Laboratório de Ciência de Dados e Inteligência Artificial*<br>
*Universidade de Fortaleza*<br>

Lattes: http://lattes.cnpq.br/5738786447903616 |
GitHub: https://github.com/carubbi/

[Unifor.br](https://unifor.br/) | [Instagram](https://www.instagram.com/uniforcomunica/?hl=pt-br) | [Facebook](https://www.facebook.com/uniforoficial/) | [Twitter](https://www.facebook.com/uniforoficial/)  | [LinkedIn](https://www.linkedin.com/school/university-of-fortaleza/?originalSubdomain=pt) | [TV Unifor](https://www.unifor.br/tv-unifor) | [G1/Ensinando e Aprendendo](https://g1.globo.com/ce/ceara/especial-publicitario/unifor/ensinando-e-aprendendo/)

# AP1 — Análise exploratória do preço de venda de imóveis residenciais

## 1. Identificação

- **Disciplina:** Métodos Quantitativos em Computação (T199)
- **Turma:** T199-64
- **Integrantes:** Adriel Medeiros Lins
- **Característica qualitativa $B$ atribuída:** Qualidade da cozinha (`Kitchen Qual`)
- **Repositório:** <https://github.com/medeiros-cs/mq-ap>

## 2. Delimitação da investigação

### 2.1 Contexto

A base reúne informações sobre imóveis residenciais, seus preços de venda e suas características. Antes de calcular probabilidades ou ajustar modelos, é necessário conhecer a estrutura, a qualidade e o comportamento dos dados.

### 2.2 Pergunta de pesquisa

Como se distribuem e variam os preços de venda de imóveis residenciais?

### 2.3 Objetivo da investigação

Caracterizar a distribuição e a variabilidade dos preços de venda e investigar suas relações com características físicas, qualitativas e temporais dos imóveis.

## 3. Carregamento dos dados

O arquivo bruto `data/raw/AmesHousing.txt`, fornecido pelo professor, será carregado em uma célula própria, preservando a cópia bruta sem alterações.

### 3.1 Importação e rastreabilidade

**Identificação e procedência do arquivo.** `AmesHousing.txt`, fornecido pelo professor Ricardo Carubbi no repositório da disciplina (`data/raw/AmesHousing.txt`). Corresponde à base *Ames Housing*, publicada por De Cock (2011) como alternativa ao conjunto *Boston Housing* para projetos de regressão. Cópia obtida em 11/09/2026 e preservada sem alterações em `data/raw/AmesHousing.txt` deste repositório.

In [ ]:
# Importar a biblioteca de manipulação de dados
import pandas as pd


#### Leitura do arquivo

In [ ]:
# Carregar o arquivo bruto fornecido pelo professor, sem aplicar transformações
df_raw = pd.read_csv("../data/raw/AmesHousing.txt", sep="\t")
df_raw.shape


**Tabela 1 - Dimensões do arquivo bruto.** O arquivo é separado por tabulação (`\t`), sem indícios de problemas de codificação (leitura padrão UTF-8 sem erros). A tabela resultante tem 2930 linhas e 82 colunas.

#### Unidade de análise

In [ ]:
# Verificar a unicidade de Order e PID para confirmar a unidade de análise
unidade_analise = pd.DataFrame({
    "Order (linhas) únicos": [df_raw["Order"].is_unique],
    "PID (imóvel) únicos": [df_raw["PID"].is_unique],
    "n_linhas": [len(df_raw)],
})
unidade_analise


**Tabela 2 - Unicidade dos identificadores.** `Order` e `PID` são ambos únicos nas 2930 linhas, confirmando que a unidade de análise é a venda individual de um imóvel residencial (um registro por transação).

#### Nomes das variáveis

In [ ]:
# Listar os nomes originais das variáveis, mantidos sem normalização nesta etapa
list(df_raw.columns)


**Nomes das variáveis.** Os nomes originais do arquivo (com espaços, ex.: `Mas Vnr Area`, `Year Remod/Add`) foram preservados sem normalização nesta etapa. Caso alguma variável específica exija um nome mais legível nas seções seguintes, o nome legível será apresentado seguido do nome original entre parênteses, conforme exigido pelo enunciado — sem alterar as colunas do `DataFrame` bruto.

### 3.2 Qualidade e preparação

#### Tipos computacionais

In [ ]:
# Verificar os tipos computacionais das colunas
df_raw.dtypes.value_counts()


**Tabela 3 - Tipos computacionais das colunas.** 43 colunas são do tipo `object` (predominantemente categóricas/texto), 28 são `int64` e 11 são `float64`. Conversões específicas serão avaliadas variável a variável, conforme necessário nas análises seguintes.

#### Duplicidades

In [ ]:
# Verificar duplicidade de linhas completas e de identificador do imóvel (PID)
pd.DataFrame({
    "linhas_duplicadas_completas": [df_raw.duplicated().sum()],
    "PID_duplicados": [df_raw["PID"].duplicated().sum()],
})


**Tabela 4 - Duplicidades.** Não há linhas completamente duplicadas nem `PID` duplicado.

#### Valores ausentes por coluna

In [ ]:
# Resumir tipo, quantidade e percentual de valores ausentes por coluna
resumo_ausentes = pd.DataFrame({
    "dtype": df_raw.dtypes,
    "n_ausentes": df_raw.isna().sum(),
    "pct_ausentes": (df_raw.isna().mean() * 100).round(1),
}).sort_values("n_ausentes", ascending=False)

resumo_ausentes.head(15)


**Tabela 5 - Valores ausentes por coluna (15 maiores).** Consultando o dicionário oficial (`data/raw/DataDocumentation.txt`), confirma-se que, em `Pool QC`, `Fence` e `Misc Feature`, o código `NA` é uma categoria válida do dicionário (respectivamente "No Pool", "No Fence" e "None"), e em `Alley`, "No alley access" — não uma ausência real, mas interpretada como tal pela leitura padrão do `pandas`. As demais colunas com poucos ausentes (ex.: `Lot Frontage`, colunas de garagem e porão) exigirão investigação específica antes de qualquer tratamento.

#### Colunas relevantes à AP2

In [ ]:
# Inspecionar tipo, ausências e intervalo das colunas relevantes à trilha Binomial da AP2
cols4 = ["Fireplaces", "Mas Vnr Area", "Year Remod/Add", "Year Built"]
inspecao_cols4 = pd.DataFrame({
    "dtype": df_raw[cols4].dtypes,
    "n_ausentes": df_raw[cols4].isna().sum(),
    "min": df_raw[cols4].min(),
    "max": df_raw[cols4].max(),
})
inspecao_cols4


**Tabela 6 - Inspeção inicial das colunas relevantes à trilha Binomial da AP2.** `Fireplaces` (`int64`, 0 a 4, sem ausências) e `Year Built`/`Year Remod/Add` (`int64`, sem ausências, intervalos plausíveis: construção entre 1872 e 2010, remodelação entre 1950 e 2010) não apresentam valores ausentes. `Mas Vnr Area` (`float64`) apresenta 23 valores ausentes, investigados a seguir.

#### Inconsistência temporal

In [ ]:
# Localizar casos em que a remodelação antecede a construção
inconsistencias_ano = df_raw[df_raw["Year Remod/Add"] < df_raw["Year Built"]]
inconsistencias_ano[["Order", "PID", "Year Built", "Year Remod/Add", "Yr Sold"]]


**Tabela 7 - Caso sinalizado de inconsistência temporal.** Uma linha (`Order` 851) registra `Year Remod/Add` (2001) anterior a `Year Built` (2002), uma inconsistência de um ano. Por afetar apenas um caso e não distorcer as análises agregadas, a linha será mantida com a inconsistência documentada; uma correção ou exclusão só será aplicada se essa observação vier a influenciar de forma relevante alguma medida ou modelo nas etapas seguintes.

#### Padrão de ausência em Mas Vnr

In [ ]:
# Comparar o padrão de ausência entre Mas Vnr Type e Mas Vnr Area
tipo_na = df_raw["Mas Vnr Type"].isna()
area_na = df_raw["Mas Vnr Area"].isna()

pd.DataFrame({
    "Tipo e área ausentes": [(tipo_na & area_na).sum()],
    "Tipo ausente, área = 0 (plausível 'sem revestimento')": [(tipo_na & ~area_na & (df_raw["Mas Vnr Area"] == 0)).sum()],
    "Tipo ausente, área > 0 (inconsistente)": [(tipo_na & ~area_na & (df_raw["Mas Vnr Area"] > 0)).sum()],
})


**Tabela 8 - Padrão de ausência em `Mas Vnr Area` e `Mas Vnr Type`.** A maior parte dos registros com `Mas Vnr Type` ausente tem `Mas Vnr Area` explicitamente igual a 0 (1745 casos), consistente com "sem revestimento em alvenaria". Os 23 registros com `Mas Vnr Area` ausente têm também `Mas Vnr Type` ausente, formando um padrão distinto dos casos de área zero — por isso, **não serão automaticamente tratados como ausência de revestimento**, conforme exigido pelo enunciado; permanecerão como valores ausentes a decidir (imputação, correção pontual ou exclusão justificada) na etapa de valores discrepantes. Adicionalmente, 7 registros têm `Mas Vnr Type` ausente com `Mas Vnr Area` maior que zero — uma inconsistência a investigar individualmente antes de qualquer tratamento.

#### Cópia de trabalho

In [ ]:
# Preservar uma cópia lógica dos dados brutos para os tratamentos das próximas etapas
df = df_raw.copy()
df.shape


**Cópia de trabalho.** A partir daqui, as etapas de tratamento e análise operam sobre `df` (cópia lógica), preservando `df_raw` intacto como registro do arquivo original.

## 4. Análise exploratória

*(Escopo do Acompanhamento 2 em diante — não iniciar antes de fechar a Seção 3.)*

### 4.1 Análise univariada

#### 4.1.1 Medidas, histogramas e gráfico de barras

In [ ]:
# Calcular medidas de posição, dispersão e forma de SalePrice e Gr Liv Area
medidas_quant = pd.DataFrame({
    "n": df[["SalePrice", "Gr Liv Area"]].count(),
    "media": df[["SalePrice", "Gr Liv Area"]].mean().round(1),
    "mediana": df[["SalePrice", "Gr Liv Area"]].median(),
    "desvio_padrao": df[["SalePrice", "Gr Liv Area"]].std().round(1),
    "minimo": df[["SalePrice", "Gr Liv Area"]].min(),
    "maximo": df[["SalePrice", "Gr Liv Area"]].max(),
    "assimetria": df[["SalePrice", "Gr Liv Area"]].skew().round(2),
    "curtose": df[["SalePrice", "Gr Liv Area"]].kurtosis().round(2),
})
medidas_quant


**Tabela 9 - Medidas de posição, dispersão e forma.** Preço de venda (`SalePrice`) tem mediana US$ 160.000 e média US$ 180.796 (maior que a mediana), com assimetria positiva de 1,74 — distribuição concentrada à esquerda, com cauda longa à direita. Área construída (`Gr Liv Area`) segue o mesmo padrão, com mediana 1.442 pés² e assimetria 1,27. Em ambas, a curtose elevada (5,12 e 4,14) indica caudas mais pesadas que as de uma distribuição normal.

In [ ]:
# Importar a biblioteca de visualização
import matplotlib.pyplot as plt


In [ ]:
# Construir histogramas de SalePrice e Gr Liv Area em um único painel
fig, eixos = plt.subplots(1, 2, figsize=(10, 4))

eixos[0].hist(df["SalePrice"], bins=30, color="#4C72B0")
eixos[0].set_xlabel("Preço de venda em dólares (SalePrice)")
eixos[0].set_ylabel("Frequência")

eixos[1].hist(df["Gr Liv Area"], bins=30, color="#55A868")
eixos[1].set_xlabel("Área construída em pés² (Gr Liv Area)")
eixos[1].set_ylabel("Frequência")

fig.tight_layout()
plt.show()


**Figura 1 - Histogramas de preço de venda e área construída.** Ambas as distribuições são unimodais e assimétricas à direita, coerente com as medidas de forma da Tabela 9: a maioria dos imóveis concentra-se em preços e áreas menores, com poucos casos de valores muito altos puxando a cauda direita.

In [ ]:
# Apresentar frequências absolutas e relativas de Kitchen Qual (qualidade da cozinha)
freq_kitchen = df["Kitchen Qual"].value_counts().to_frame("frequencia_absoluta")
freq_kitchen["frequencia_relativa_pct"] = (freq_kitchen["frequencia_absoluta"] / len(df) * 100).round(1)
freq_kitchen


**Tabela 10 - Frequências de qualidade da cozinha (`Kitchen Qual`).** As categorias `TA` (51,0%) e `Gd` (39,6%) concentram mais de 90% dos imóveis. `Po` tem apenas 1 observação (0,03%) — uma categoria rara demais para sustentar qualquer comparação isolada, tratada na Seção 4.1.3.

In [ ]:
# Construir o gráfico de barras de Kitchen Qual, preservando a ordem de qualidade
ordem_kitchen = ["Po", "Fa", "TA", "Gd", "Ex"]
contagem_kitchen = df["Kitchen Qual"].value_counts().reindex(ordem_kitchen)

fig, eixo = plt.subplots(figsize=(6, 4))
eixo.bar(contagem_kitchen.index, contagem_kitchen.to_numpy(), color="#C44E52")
eixo.set_xlabel("Qualidade da cozinha (Kitchen Qual)")
eixo.set_ylabel("Quantidade de imóveis")
eixo.set_title("Distribuição da qualidade da cozinha (Kitchen Qual)")
plt.show()


**Figura 2 - Qualidade da cozinha (Kitchen Qual).** O gráfico confirma visualmente a concentração em `TA` e `Gd` e a quase ausência de imóveis em `Po`, reforçando a necessidade de agrupamento antes de qualquer comparação de preço por categoria.

#### 4.1.2 Valores discrepantes (cercas de Tukey)

In [ ]:
# Calcular as cercas de Tukey e contar os casos sinalizados
def cercas_tukey(serie):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return pd.Series({
        "Q1": q1, "Q3": q3, "IQR": iqr,
        "limite_inferior": li, "limite_superior": ls,
        "n_sinalizados": ((serie < li) | (serie > ls)).sum(),
    })

tabela_tukey = df[["SalePrice", "Gr Liv Area"]].apply(cercas_tukey).T
tabela_tukey


**Tabela 11 - Cercas de Tukey.** Para `SalePrice`, o limite superior é US$ 339.500, com 137 casos sinalizados (todos acima do limite; o limite inferior é negativo na prática, US$ 3.500). Para `Gr Liv Area`, o limite superior é 2.668 pés², com 75 casos sinalizados.

In [ ]:
# Construir os boxplots de SalePrice e Gr Liv Area em um único painel
fig, eixos = plt.subplots(1, 2, figsize=(8, 4))

eixos[0].boxplot(df["SalePrice"], tick_labels=["SalePrice"])
eixos[0].set_ylabel("Preço de venda em dólares (SalePrice)")

eixos[1].boxplot(df["Gr Liv Area"], tick_labels=["Gr Liv Area"])
eixos[1].set_ylabel("Área construída em pés² (Gr Liv Area)")

fig.tight_layout()
plt.show()


**Figura 3 - Boxplots de preço de venda e área construída.** Os dois boxplots mostram caudas superiores longas com diversos pontos além do limite superior, consistente com a Tabela 11 e com a assimetria positiva já observada na Figura 1.

In [ ]:
# Investigar os casos sinalizados por condição de venda (Sale Condition)
flag_price = (df["SalePrice"] < tabela_tukey.loc["SalePrice", "limite_inferior"]) | (df["SalePrice"] > tabela_tukey.loc["SalePrice", "limite_superior"])
flag_area = (df["Gr Liv Area"] < tabela_tukey.loc["Gr Liv Area", "limite_inferior"]) | (df["Gr Liv Area"] > tabela_tukey.loc["Gr Liv Area", "limite_superior"])

investigacao_sinalizados = pd.DataFrame({
    "sinalizados_SalePrice": df.loc[flag_price, "Sale Condition"].value_counts(),
    "sinalizados_Gr_Liv_Area": df.loc[flag_area, "Sale Condition"].value_counts(),
    "base_completa": df["Sale Condition"].value_counts(),
}).fillna(0).astype(int)
investigacao_sinalizados


**Tabela 12 - Casos sinalizados por condição de venda.** `Partial` (venda de imóvel novo, geralmente ainda em construção) responde por 42% dos casos sinalizados em `SalePrice` (58 de 137) e 21% em `Gr Liv Area` (16 de 75), ante apenas 8% na base completa — uma sobrerrepresentação esperada, já que imóveis novos tendem a ser maiores e mais caros. Isso indica que a maior parte dos casos sinalizados reflete vendas legítimas de um segmento de mercado específico, não erros de registro.

In [ ]:
# Localizar os cinco casos de área extrema já documentados no artigo de referência (De Cock, 2011)
df[df["Gr Liv Area"] > 4000][["Order", "PID", "Gr Liv Area", "SalePrice", "Sale Condition"]]


**Tabela 13 - Casos de área extrema documentados por De Cock (2011).** As cinco observações com `Gr Liv Area` acima de 4.000 pés² são exatamente as apontadas pelo autor da base como atípicas: três são vendas `Partial` (imóvel novo, preço ainda não totalmente refletido) e uma é `Abnorml`, todas com preço desproporcionalmente baixo para a área.

**Decisão sobre os casos sinalizados.** Nenhum caso será excluído nesta etapa: a investigação mostra que a maioria dos valores sinalizados corresponde a vendas legítimas (imóveis grandes ou de alto padrão, muitos deles `Partial`), não a erros de digitação ou medição. Os cinco casos da Tabela 13 são mantidos, mas ficam documentados como pontos de alta influência a serem reavaliados no diagnóstico de regressão da AP3, quando uma eventual exclusão orientada por resíduos e alavancagem for mais apropriada do que uma remoção univariada nesta etapa exploratória.

#### 4.1.3 Característica qualitativa B — agrupamento final

In [ ]:
# Agrupar a categoria rara Po com Fa, por semelhança de significado e de preço mediano
df["Kitchen Qual Final"] = df["Kitchen Qual"].replace({"Po": "Fa/Po", "Fa": "Fa/Po"})

pd.DataFrame({
    "antes": df["Kitchen Qual"].value_counts().reindex(["Po", "Fa", "TA", "Gd", "Ex"]),
    "depois": df["Kitchen Qual Final"].value_counts().reindex(["Fa/Po", "TA", "Gd", "Ex"]),
})


**Tabela 14 - Agrupamento final de qualidade da cozinha (Kitchen Qual → Kitchen Qual Final).** `Po` (n=1) foi unido a `Fa` (n=70), formando a categoria `Fa/Po` (n=71). A junção é justificada pelo significado — ambas as categorias indicam qualidade abaixo da média — e pela proximidade do preço mediano de venda entre os dois grupos originais (US$ 107.500 em `Po` e US$ 107.750 em `Fa`). As categorias finais — `Ex`, `Gd`, `TA`, `Fa/Po` — têm todas n ≥ 71, permitindo comparação. Esta decisão fica registrada e preservada para a AP3.

### 4.2 Análise bivariada

#### 4.2.1 Preço de venda e ano da venda

In [ ]:
# Calcular a quantidade de vendas e a mediana do preço por ano
preco_por_ano = df.groupby("Yr Sold")["SalePrice"].agg(n_vendas="count", mediana_SalePrice="median")
preco_por_ano


**Tabela 15 - Vendas e mediana anual do preço.** A mediana varia pouco entre 2006 e 2010 (entre US$ 155.000 e US$ 165.125), sem uma tendência clara de alta ou queda. O ano de 2010 tem apenas 341 vendas, bem menos que os demais (622 a 694) — a base provavelmente cobre 2010 parcialmente, o que limita a comparação com esse ano.

In [ ]:
# Construir o gráfico de linha da mediana anual do preço de venda, em ordem cronológica
fig, eixo = plt.subplots(figsize=(6, 4))
eixo.plot(preco_por_ano.index, preco_por_ano["mediana_SalePrice"], marker="o")
eixo.set_xlabel("Ano da venda (Yr Sold)")
eixo.set_ylabel("Mediana do preço de venda em dólares (SalePrice)")
eixo.set_xticks(preco_por_ano.index)
plt.show()


**Figura 4 - Mediana anual do preço de venda.** A linha liga apenas as medianas anuais (não preços de imóveis individuais) e não deve ser lida como tendência de mercado, trajetória de um mesmo imóvel ou série temporal formal: as variações entre anos podem refletir mudanças na composição dos imóveis vendidos, inflação e condições de mercado (a base cobre justamente o período da crise financeira de 2008), não apenas o valor de mercado em si.

#### 4.2.2 Preço de venda e área construída

In [ ]:
# Construir o diagrama de dispersão entre SalePrice e Gr Liv Area
fig, eixo = plt.subplots(figsize=(6, 4))
eixo.scatter(df["Gr Liv Area"], df["SalePrice"], alpha=0.3, s=10)
eixo.set_xlabel("Área construída em pés² (Gr Liv Area)")
eixo.set_ylabel("Preço de venda em dólares (SalePrice)")
plt.show()


In [ ]:
# Calcular a correlação linear de Pearson entre SalePrice e Gr Liv Area
df["SalePrice"].corr(df["Gr Liv Area"])


**Figura 5 - Dispersão entre preço de venda e área construída.** A relação é positiva, moderadamente forte e aproximadamente linear (correlação de Pearson r ≈ 0,71): imóveis maiores tendem a ter preços mais altos. Os cinco casos de área extrema da Tabela 13 aparecem isolados à direita, com preço desproporcionalmente baixo para a área — eles puxam o coeficiente para baixo e ilustram como valores discrepantes podem distorcer a correlação. A associação não implica causalidade: outras características correlacionadas com a área (localização, padrão construtivo) também afetam o preço.

#### 4.2.3 Preço de venda por categoria de B

In [ ]:
# Apresentar quantidade, mediana, Q1, Q3 e IQR do preço por categoria final de Kitchen Qual
def resumo_preco(grupo):
    q1, q3 = grupo.quantile(0.25), grupo.quantile(0.75)
    return pd.Series({"n": grupo.count(), "mediana": grupo.median(), "Q1": q1, "Q3": q3, "IQR": q3 - q1})

ordem_final = ["Fa/Po", "TA", "Gd", "Ex"]
preco_por_kitchen = df.groupby("Kitchen Qual Final")["SalePrice"].apply(resumo_preco).unstack().reindex(ordem_final)
preco_por_kitchen


**Tabela 16 - Preço de venda por qualidade da cozinha (Kitchen Qual Final).** A mediana cresce de forma monotônica com a qualidade: US$ 107.500 (`Fa/Po`), US$ 136.500 (`TA`), US$ 201.000 (`Gd`) e US$ 325.624 (`Ex`). O IQR também aumenta nas categorias mais caras, indicando maior dispersão de preços entre imóveis com cozinha de alta qualidade.

In [ ]:
# Construir os boxplots do preço de venda por categoria final de Kitchen Qual
fig, eixo = plt.subplots(figsize=(6, 4))
dados_boxplot = [df.loc[df["Kitchen Qual Final"] == cat, "SalePrice"] for cat in ordem_final]
eixo.boxplot(dados_boxplot, tick_labels=ordem_final)
eixo.set_xlabel("Qualidade da cozinha (Kitchen Qual Final)")
eixo.set_ylabel("Preço de venda em dólares (SalePrice)")
plt.show()


**Figura 6 - Preço de venda por qualidade da cozinha (Kitchen Qual Final).** Os boxplots mostram posições nitidamente diferentes entre categorias e dispersão crescente nas categorias de maior qualidade, com vários valores discrepantes acima do bigode superior — sobretudo em `Gd` e `Ex`. As diferenças são apenas descritivas: nenhuma inferência estatística ou afirmação causal é feita nesta etapa.

## 5. Exportação dos dados

**Registro de nomes e tratamentos.** Os nomes originais das colunas foram preservados sem normalização (Seção 3.1). Os tratamentos aplicados foram: manutenção da inconsistência pontual entre `Year Built` e `Year Remod/Add` no registro `Order` 851 (Seção 3.2); preservação dos 23 valores ausentes de `Mas Vnr Area` sem preenchimento automático (Seção 3.2); manutenção de todos os casos sinalizados pelas cercas de Tukey em `SalePrice` e `Gr Liv Area`, sem exclusão (Seção 4.1.2).

**Agrupamento final de $B$.** `Kitchen Qual` (qualidade da cozinha) foi agrupada em `Kitchen Qual Final`, unindo `Po` a `Fa` na categoria `Fa/Po` (Seção 4.1.3). As categorias finais são `Fa/Po`, `TA`, `Gd` e `Ex`.

**Mapeamento de nomes.** Nenhuma coluna analisada (`SalePrice`, `Gr Liv Area`, `Yr Sold`, `Kitchen Qual`) foi renomeada; apenas `Kitchen Qual Final` foi adicionada como nova coluna derivada.

**Reprodução.** O arquivo processado é gerado executando este notebook do início ao fim a partir de `data/raw/AmesHousing.txt`; nenhuma edição manual é aplicada ao resultado.

In [ ]:
# Exportar o dado pré-processado de forma reproduzível
df.to_csv("../data/processed/AmesHousing.csv", index=False)


In [ ]:
# Verificar a reprodutibilidade do arquivo exportado
pd.read_csv("../data/processed/AmesHousing.csv").shape


**Verificação.** O arquivo `data/processed/AmesHousing.csv` é recarregado com 2930 linhas e as mesmas colunas de `df`, incluindo `Kitchen Qual Final`, confirmando a reprodutibilidade do processo.

## 6. Conclusão e limitações

Os preços de venda dos imóveis de Ames são fortemente assimétricos à direita, concentrados em torno de US$ 160 mil, com poucos imóveis de alto valor esticando a distribuição. Área construída e qualidade da cozinha (`Kitchen Qual`) associam-se positivamente ao preço — quanto maior a área e melhor a qualidade da cozinha, maior a mediana do preço —, mas nenhuma dessas relações foi tratada como causal.

**Limitações.**

- **Natureza observacional.** Os dados vêm de registros administrativos do escritório de avaliação de Ames (De Cock, 2011), não de um experimento controlado; nenhuma associação encontrada implica intervenção ou manipulação de variáveis.
- **Período e local.** A base cobre apenas o município de Ames, Iowa, entre 2006 e 2010 — um intervalo que inclui a crise financeira de 2008 — e não deve ser generalizada para outras cidades, países ou períodos.
- **Representatividade.** Mesmo dentro de Ames, a base reflete apenas os imóveis efetivamente vendidos no período, não o estoque completo de imóveis residenciais da cidade.
- **Efeito das decisões de preparação.** A inconsistência de um ano entre `Year Built` e `Year Remod/Add` (`Order` 851) foi mantida; os 23 valores ausentes de `Mas Vnr Area` permanecem sem preenchimento; nenhum caso sinalizado pelas cercas de Tukey foi excluído. Cada uma dessas escolhas foi documentada e pode ser revisitada nas próximas etapas.
- **Discrepante não é sinônimo de erro.** A maior parte dos casos sinalizados em `SalePrice` e `Gr Liv Area` corresponde a vendas legítimas de imóveis grandes, novos (`Partial`) ou de alto padrão — não a erros de registro ou medição.
- **Associação não é causalidade.** A correlação entre `SalePrice` e `Gr Liv Area` (r ≈ 0,71) e as diferenças de preço por `Kitchen Qual Final` são descritivas; nenhum modelo estatístico foi ajustado nesta etapa, e nenhuma das relações reportadas permite afirmar causalidade.

## 7. Contribuições dos integrantes

TODO: descrever a contribuição de cada integrante.

## 8. Declaração de uso de inteligência artificial

TODO: ferramenta/modelo, finalidade, etapas afetadas e verificação do conteúdo (ou declaração de não uso).

## 9. Referências

DE COCK, Dean. Ames, Iowa: alternative to the Boston housing data as an end of semester regression project. *Journal of Statistics Education*, v. 19, n. 3, 2011. Disponível em: <https://jse.amstat.org/v19n3/decock.pdf>.

DE COCK, Dean. *Ames Housing dataset — data documentation* (`DataDocumentation.txt`). Dicionário oficial de variáveis e códigos, fornecido junto ao arquivo bruto (`data/raw/DataDocumentation.txt`). Consultado para interpretar os códigos de variáveis ordinais/nominais, incluindo `KitchenQual` (característica $B$).